# Analyse de complexité — Problème de transport

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

## Chargement de tous les fichiers CSV

In [ ]:
# Récupère tous les CSV du dossier complexite
fichiers = glob.glob('complexite/*.csv')

# Vérification
if not fichiers:
    raise FileNotFoundError(
        "Aucun fichier CSV trouvé dans le dossier 'complexite'"
    )

# Lecture + concaténation
dfs = []

for fichier in fichiers:
    df_temp = pd.read_csv(fichier)

    # Ajoute le nom du fichier pour traçabilité
    df_temp['source'] = os.path.basename(fichier)

    dfs.append(df_temp)

# Fusion de tous les fichiers
df = pd.concat(dfs, ignore_index=True)

print(f"{len(fichiers)} fichiers chargés")
print(f"{len(df)} lignes au total")

# Aperçu
print(df.head())

## Préparation des données

In [ ]:
# Enveloppe supérieure du nuage de points
pire_des_cas = df.groupby('n').max(numeric_only=True)

print(pire_des_cas.head())

## Métriques étudiées

In [ ]:
metriques = {
    'thetaNO_ns': 'Temps Nord-Ouest (theta_NO)',
    'thetaBH_ns': 'Temps Balas-Hammer (theta_BH)',
    'tNO_ns': 'Temps Marche-pied depuis NO (t_NO)',
    'tBH_ns': 'Temps Marche-pied depuis BH (t_BH)',
    'totalNO_ns': 'Total (theta_NO + t_NO)',
    'totalBH_ns': 'Total (theta_BH + t_BH)'
}

## Nuage de points global + enveloppe supérieure

In [ ]:
plt.figure(figsize=(12, 9))

for col, label in metriques.items():

    # Nuage de points
    plt.scatter(
        df['n'],
        df[col],
        alpha=0.20,
        s=12,
        label=f'Données {label}'
    )

    # Pire des cas
    plt.plot(
        pire_des_cas.index,
        pire_des_cas[col],
        marker='o',
        linestyle='--'
    )

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Taille du problème (n)')
plt.ylabel('Temps (ns)')

plt.title(
    'Analyse de la complexité temporelle\n'
    '(Tous fichiers fusionnés)'
)

plt.legend(fontsize=8)

plt.grid(
    True,
    which="both",
    ls="-",
    alpha=0.2
)

plt.tight_layout()
plt.show()

## Comparaison directe des algorithmes complets

In [ ]:
plt.figure(figsize=(12, 9))

plt.plot(
    pire_des_cas.index,
    pire_des_cas['totalNO_ns'],
    marker='s',
    linewidth=2,
    label='Pire des cas total NO'
)

plt.plot(
    pire_des_cas.index,
    pire_des_cas['totalBH_ns'],
    marker='s',
    linewidth=2,
    label='Pire des cas total BH'
)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Taille du problème (n)')
plt.ylabel('Temps total (ns)')

plt.title(
    'Comparaison Nord-Ouest vs Balas-Hammer\n'
    '(Tous fichiers fusionnés)'
)

plt.legend()

plt.grid(
    True,
    which="both",
    ls="-",
    alpha=0.2
)

plt.tight_layout()
plt.show()

## Analyse complémentaire : temps moyens

In [ ]:
moyennes = df.groupby('n').mean(numeric_only=True)

plt.figure(figsize=(12, 9))

plt.plot(
    moyennes.index,
    moyennes['totalNO_ns'],
    marker='o',
    label='Temps moyen total NO'
)

plt.plot(
    moyennes.index,
    moyennes['totalBH_ns'],
    marker='o',
    label='Temps moyen total BH'
)

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Taille du problème (n)')
plt.ylabel('Temps moyen (ns)')

plt.title('Temps moyens des algorithmes')

plt.legend()

plt.grid(True, which="both", alpha=0.2)

plt.tight_layout()
plt.show()

## Nombre de cas ayant atteint maxIter

In [ ]:
if 'optimalNO' in df.columns and 'optimalBH' in df.columns:

    no_fail = (~df['optimalNO']).sum()
    bh_fail = (~df['optimalBH']).sum()

    print(f"Cas maxIter atteints (NO) : {no_fail}")
    print(f"Cas maxIter atteints (BH) : {bh_fail}")